<a href="https://colab.research.google.com/github/Adyypower/Deep-learning-Models-or-topics/blob/main/summerize_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install transformers datasets accelerate peft evaluate loralib -q

print("✅ All necessary libraries have been installed.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 8.0 MB/s eta 0:00:00
✅ All necessary libraries have been installed.


In [5]:
from datasets import load_dataset

# Load the dataset using the alternative path you found
dataset = load_dataset("knkarthick/samsum")

print("✅ Dataset loaded successfully using the alternative path!")
print("\nHere's the structure of the dataset:")
print(dataset)

print("\nHere is one example from the training set:")
print(dataset['train'][0])

README.md: 0.00B [00:00, ?B/s]

train.csv: 0.00B [00:00, ?B/s]

validation.csv: 0.00B [00:00, ?B/s]

test.csv: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/14731 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/818 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/819 [00:00<?, ? examples/s]

✅ Dataset loaded successfully using the alternative path!

Here's the structure of the dataset:
DatasetDict({
    train: Dataset({
        features: ['id', 'dialogue', 'summary'],
        num_rows: 14731
    })
    validation: Dataset({
        features: ['id', 'dialogue', 'summary'],
        num_rows: 818
    })
    test: Dataset({
        features: ['id', 'dialogue', 'summary'],
        num_rows: 819
    })
})

Here is one example from the training set:
{'id': '13818513', 'dialogue': "Amanda: I baked  cookies. Do you want some?\nJerry: Sure!\nAmanda: I'll bring you tomorrow :-)", 'summary': 'Amanda baked cookies and will bring Jerry some tomorrow.'}


In [26]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
model_name = "google/flan-t5-small"
tokenizer = AutoTokenizer.from_pretrained(model_name)

def preprocess_function(examples):
  inputs = [f"Summerize the following conversation.\n\n{dialogue}\n\nsummary:" for dialogue in examples["dialogue"]]
  model_inputs = tokenizer(inputs, max_length=1024, truncation=True)

  labels = tokenizer(text_target = examples["summary"],max_length = 128, truncation = True)

  model_inputs["labels"] = labels["input_ids"]
print("Aplying the preprocessing function to the dataset...")

tokenizer_dataset = dataset.map(preprocess_function,batched = True)
print("✅ Preprocessing done!")

print(tokenizer_dataset)

Aplying the preprocessing function to the dataset...


Map:   0%|          | 0/14731 [00:00<?, ? examples/s]

Map:   0%|          | 0/818 [00:00<?, ? examples/s]

Map:   0%|          | 0/819 [00:00<?, ? examples/s]

✅ Preprocessing done!
DatasetDict({
    train: Dataset({
        features: ['id', 'dialogue', 'summary'],
        num_rows: 14731
    })
    validation: Dataset({
        features: ['id', 'dialogue', 'summary'],
        num_rows: 818
    })
    test: Dataset({
        features: ['id', 'dialogue', 'summary'],
        num_rows: 819
    })
})


In [15]:
from transformers import AutoModelForSeq2SeqLM
from peft import get_peft_model, LoraConfig, TaskType

# 1. Define the PEFT (LoRA) configuration
lora_config = LoraConfig(
    r=8, # Rank of the update matrices. Lower means less trainable parameters.
    lora_alpha=32, # Alpha parameter for scaling LoRA weights.
    lora_dropout=0.1, # Dropout probability for LoRA layers.
    bias="none",
    task_type=TaskType.SEQ_2_SEQ_LM # This is crucial for T5 models.
)

# 2. Load the base Flan-T5 model
model_name = "google/flan-t5-small"
base_model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

# 3. Apply the LoRA configuration to the base model
peft_model = get_peft_model(base_model, lora_config)

# 4. Print the percentage of trainable parameters
# This shows how efficient LoRA is!
def print_trainable_parameters(model):
    """
    Prints the number of trainable parameters in the model.
    """
    trainable_params = 0
    all_param = 0
    for _, param in model.named_parameters():
        all_param += param.numel()
        if param.requires_grad:
            trainable_params += param.numel()
    print(
        f"trainable params: {trainable_params} || all params: {all_param} || trainable%: {100 * trainable_params / all_param:.2f}"
    )

print("Model parameters before applying PEFT:")
print_trainable_parameters(base_model)

print("\nModel parameters after applying PEFT (LoRA):")
print_trainable_parameters(peft_model)

print("\n✅ Model is loaded and configured with PEFT.")

Model parameters before applying PEFT:
trainable params: 344064 || all params: 77305216 || trainable%: 0.45

Model parameters after applying PEFT (LoRA):
trainable params: 344064 || all params: 77305216 || trainable%: 0.45

✅ Model is loaded and configured with PEFT.


In [25]:
from transformers import TrainingArguments, Trainer, DataCollatorForSeq2Seq
import os
os.environ["WANDB_DISABLED"] = "true"

# 1. Define the training arguments
# These settings control the training process
training_args = TrainingArguments(
    output_dir="./t5-samsum-summary-model", # Where the model will be saved
    learning_rate=3e-4,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    num_train_epochs=1, # We'll train for one full pass through the data
    weight_decay=0.01,
    load_best_model_at_end=True, # Load the best version of the model at the end
    eval_strategy="epoch", # Set evaluation strategy to epoch
    save_strategy="epoch", # Set save strategy to epoch
    remove_unused_columns=False, # Keep unused columns - This might not be needed if we explicitly select columns
)

# 2. Initialize the Data Collator
data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=peft_model)

# 3. Prepare the datasets for training and evaluation by removing the unused columns
# We will remove the original columns that are not needed for training
# cols_to_remove = ['id', 'dialogue', 'summary'] # These columns are already removed during preprocessing
train_dataset = tokenized_dataset["train"] # Use the tokenized dataset directly
eval_dataset = tokenized_dataset["validation"] # Use the tokenized dataset directly


# 4. Initialize the Trainer
trainer = Trainer(
    model=peft_model,
    args=training_args,
    train_dataset=train_dataset, # Use the dataset with removed columns
    eval_dataset=eval_dataset, # Use the dataset with removed columns
    tokenizer=tokenizer,
    data_collator=data_collator, # Add the data collator here
)

# 5. Start the fine-tuning! 🚀
print("Starting the fine-tuning process...")
trainer.train()

print("\n🎉🎉🎉 Fine-tuning complete! 🎉🎉🎉")

Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).
/tmp/ipython-input-32894362.py:31: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Starting the fine-tuning process...


Epoch,Training Loss,Validation Loss
1,1.782700,1.675599



🎉🎉🎉 Fine-tuning complete! 🎉🎉🎉


In [31]:
# Check the columns of the processed training data
print(tokenizer_dataset["train"].column_names)

['id', 'dialogue', 'summary']


In [28]:
import torch

# Select an example from the test set
index = 15 # You can change this index to test different conversations
test_example = dataset["test"][index]

# Prepare the prompt
prompt = f"Summarize the following conversation.\n\n{test_example['dialogue']}\n\nSummary:"

# Tokenize the input, keeping the entire output dictionary
inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

# 1. Generate the summary
# Use **inputs to unpack the dictionary into keyword arguments (e.g., input_ids=..., attention_mask=...)
output_tokens = peft_model.generate(**inputs, max_new_tokens=64)

# 2. Decode the output
generated_summary = tokenizer.decode(output_tokens[0], skip_special_tokens=True)

# 3. Print the results for comparison
print("====================================================================")
print("ORIGINAL DIALOGUE:")
print(test_example['dialogue'])
print("\n====================================================================")
print("ORIGINAL HUMAN SUMMARY:")
print(test_example['summary'])
print("\n====================================================================")
print("MODEL-GENERATED SUMMARY:")
print(generated_summary)
print("====================================================================")

ORIGINAL DIALOGUE:
Greg: Hi, honey. I need to stay after hours :-(
Betsy: Again?
Greg: I'm sorry!
Betsy: What about Johnny?
Greg: Well, could you pick him up? 
Betsy: What if I can't?
Greg: Betsy?
Betsy: What if I can't?
Greg: Can't you, really?
Betsy: I can't. Today I need to work long hours as well. Tuesdays are your days in the kindergarten.
Greg: Talk to you later. I'll see what I can do.
Betsy: You'd better think of something.
Greg: Oh. Just stop it now.

ORIGINAL HUMAN SUMMARY:
Greg and Betsy have a lot of work today, so they cannot pick up Johnny from the kindergarten. However, it's Greg's turn to do it. Greg will try to find a solution.

MODEL-GENERATED SUMMARY:
Greg needs to stay after hours. He will pick Johnny up. He will talk to Greg later.


In [29]:
import torch

# 1. Define your own conversation data
my_own_conversation = """
Alex: Hey, are you free this weekend? I was thinking of going for a hike.
Ben: Oh, that sounds great! I'm busy on Saturday, but Sunday is perfect.
Alex: Awesome. What about the Blue Ridge trail? It's about a 2-hour hike.
Ben: I've heard good things about it. What time should we meet?
Alex: Let's meet at the trailhead at 9 AM. I'll bring snacks.
Ben: Perfect, see you on Sunday at 9!
"""

# 2. Prepare the prompt
prompt = f"Summarize the following conversation.\n\n{my_own_conversation}\n\nSummary:"

# 3. Tokenize the input and move to the GPU
inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

# 4. Generate the summary
output_tokens = peft_model.generate(**inputs, max_new_tokens=64)

# 5. Decode and print the result
generated_summary = tokenizer.decode(output_tokens[0], skip_special_tokens=True)

print("====================================================================")
print("YOUR CONVERSATION:")
print(my_own_conversation)
print("\n====================================================================")
print("MODEL-GENERATED SUMMARY:")
print(generated_summary)
print("====================================================================")

YOUR CONVERSATION:

Alex: Hey, are you free this weekend? I was thinking of going for a hike.
Ben: Oh, that sounds great! I'm busy on Saturday, but Sunday is perfect.
Alex: Awesome. What about the Blue Ridge trail? It's about a 2-hour hike.
Ben: I've heard good things about it. What time should we meet?
Alex: Let's meet at the trailhead at 9 AM. I'll bring snacks.
Ben: Perfect, see you on Sunday at 9!


MODEL-GENERATED SUMMARY:
Alex and Ben are going for a hike on Saturday. They will meet at the trailhead at 9 AM.
